# 03. 토큰 사용량 확인하기

요청 1회에 토큰을 얼마나 썼는지 확인한다. 요금은 이 숫자로 계산된다.

## 먼저 알아둘 것

- 모든 응답에는 `response.usage`가 함께 온다. 별도 요청 없이 바로 읽으면 된다.
- **Total Token 필드는 따로 없다.** 직접 더해서 계산한다.
- **입력과 출력의 단가가 다르다.** `claude-opus-5-5` 기준 입력 $4 / 출력 $20 (100만 토큰당). 그래서 합계보다 입력/출력을 나눠 보는 것이 중요하다.
- 한국어는 대략 **1글자(공백 포함) ≈ 1토큰**, 영어는 약 3.5글자 ≈ 1토큰.

## usage 필드

| 필드 | 의미 |
|---|---|
| `input_tokens` | 보낸 입력 토큰 (system + messages 전체) |
| `output_tokens` | 모델이 생성한 토큰. **thinking + 답변**을 합친 값 |
| `output_tokens_details.thinking_tokens` | 출력 토큰 중 thinking에 쓴 토큰 |
| `cache_creation_input_tokens` | 프롬프트 캐시에 새로 저장한 입력 토큰 (캐싱을 쓸 때만) |
| `cache_read_input_tokens` | 캐시에서 읽은 입력 토큰 (캐싱을 쓸 때만) |

캐싱을 쓰지 않으면 cache 필드 두 개는 0이다. 캐싱을 쓰면 입력이 세 칸으로 나뉘므로, 전체 입력은 세 값을 더해야 한다.

In [1]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

MODEL = "claude-opus-5-5"

# 100만 토큰당 요금 (USD)
INPUT_PRICE = 4.00
OUTPUT_PRICE = 20.00

## 1. 요청 1회 보내기

In [2]:
response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[
        {"role": "user", "content": "파이썬에서 딕셔너리를 순회하는 방법 세 가지를 짧게 알려줘."}
    ],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

파이썬에서 딕셔너리를 순회하는 대표적인 방법 세 가지입니다.

```python
d = {"a": 1, "b": 2, "c": 3}
```

**1. 키 순회 (기본)**
```python
for key in d:          # d.keys()와 동일
    print(key, d[key])
```

**2. 값만 순회**
```python
for value in d.values():
    print(value)
```

**3. 키와 값을 함께 순회**
```python
for key, value in d.items():
    print(key, value)
```

키와 값이 모두 필요하면 `items()`를 쓰는 것이 가장 깔끔합니다.


## 2. Input / Output / Total 토큰 확인

In [3]:
usage = response.usage

# 캐싱을 쓰지 않으면 cache 값은 0 (또는 None)이므로 `or 0`으로 처리
input_tokens = (
    usage.input_tokens
    + (usage.cache_creation_input_tokens or 0)
    + (usage.cache_read_input_tokens or 0)
)
output_tokens = usage.output_tokens
total_tokens = input_tokens + output_tokens

print(f"Input  : {input_tokens:>6,} tokens")
print(f"Output : {output_tokens:>6,} tokens")
print(f"Total  : {total_tokens:>6,} tokens")

# 캐싱이 없다는 가정의 단순 계산 (캐시 저장/읽기는 단가가 다름)
cost = input_tokens * INPUT_PRICE / 1_000_000 + output_tokens * OUTPUT_PRICE / 1_000_000
print(f"\n예상 요금: ${cost:.5f}")

Input  :     46 tokens
Output :    252 tokens
Total  :    298 tokens

예상 요금: $0.00522


`usage` 객체를 통째로 출력하면 위 표에 없는 필드도 볼 수 있다. 대부분 지금은 신경 쓰지 않아도 되는 부가 정보다.

In [4]:
print(usage)

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=46, output_tokens=252, output_tokens_details=OutputTokensDetails(thinking_tokens=30), server_tool_use=None, service_tier='standard', speed='standard')


## 3. 보내기 전에 입력 토큰 미리 세기 (무료)

`client.messages.count_tokens()`는 모델을 실행하지 않고 입력 토큰 수만 세어준다.

- 요금이 발생하지 않는다. (요청 수 제한은 적용됨)
- 긴 문서를 보내기 전에 컨텍스트 한도를 넘지 않는지, 비용이 얼마나 될지 확인할 때 쓴다.
- **출력 토큰은 미리 알 수 없다.** 모델이 생성해 봐야 안다.

In [5]:
messages = [
    {"role": "user", "content": "파이썬에서 딕셔너리를 순회하는 방법 세 가지를 짧게 알려줘."}
]

count = client.messages.count_tokens(model=MODEL, messages=messages)

print(f"count_tokens로 센 입력 토큰 : {count.input_tokens}")
print(f"실제 요청의 입력 토큰       : {response.usage.input_tokens}")

count_tokens로 센 입력 토큰 : 46
실제 요청의 입력 토큰       : 46


> 보통 두 값은 같거나 거의 같다. 공식 문서상 `count_tokens`는 **추정치**이므로, 기능(도구 사용 등)에 따라 몇 토큰 차이가 날 수 있다.

## 4. 출력 토큰 중 thinking은 얼마나 될까?

`output_tokens`는 thinking과 답변을 합친 값이다. 그중 thinking에 쓴 양은 `usage.output_tokens_details.thinking_tokens`에 따로 들어 있다.
화면에 보이지 않는 thinking도 **출력 단가로 과금**되므로, 출력 토큰이 답변 길이보다 많아 보이면 이 값을 확인한다.

In [6]:
details = response.usage.output_tokens_details
thinking_tokens = (details.thinking_tokens if details else 0) or 0
answer_tokens = response.usage.output_tokens - thinking_tokens

print(f"전체 출력     : {response.usage.output_tokens:>6,} tokens")
print(f"  thinking    : {thinking_tokens:>6,} tokens")
print(f"  답변        : {answer_tokens:>6,} tokens")

전체 출력     :    252 tokens
  thinking    :     30 tokens
  답변        :    222 tokens


## 정리

- 요금은 **입력 토큰 × 입력 단가 + 출력 토큰 × 출력 단가**
- 출력 단가가 5배 비싸므로, 비용에 더 큰 영향을 주는 것은 보통 **출력 토큰**이다.
- 출력 토큰을 줄이는 방법
  - 프롬프트에 원하는 길이를 명시한다 ("세 줄로", "200자 이내로")
  - 간단한 작업은 `output_config={"effort": "low"}`로 thinking을 줄인다
- 입력 토큰을 줄이는 방법
  - 멀티턴 대화가 길어지면 입력이 계속 늘어난다 → 프롬프트 캐싱으로 반복 부분을 저렴하게 처리